# Lecture 5 — Text Analysis with LLMs

This notebook demonstrates how to perform basic text analysis tasks using a large language
model (LLM). We cover sentiment analysis and text classification applied to small
example datasets.

The examples use a local LLM running through [Ollama](https://ollama.com/), which is free
and keeps all data on your machine. At the end, we also show how to connect to cloud APIs
(e.g. Gemini) as an alternative.

Why? Because free tokens on Gemini are not enough to run even 

**Credits:**

Created by Sippo Rossi for the course Python Programming for Business Intelligence at Hanken.

## 1. Setting up a local LLM with Ollama

### What is Ollama?

[Ollama](https://ollama.com/) is a tool for downloading and running open-source large
language models locally on your own computer. Instead of sending your data to a cloud
service (like OpenAI, Google, or Anthropic), the model runs entirely on your machine.

This has several advantages:
- **Free:** No API costs or token limits. You can run as many prompts as you want.
- **Private:** Your data never leaves your computer, which matters for sensitive business data.
- **Offline:** Works without an internet connection once the model is downloaded.

The trade-off is that local models are typically smaller and (much) less capable than frontier
cloud models (e.g. GPT-4, Gemini, Claude), and they require sufficient hardware to run.
Most modern laptops with 8 GB (preferably 16 GB) of RAM can run smaller models (1–4 billion parameters)
comfortably. A GPU helps with speed but is not required.

### 1.1 Installing Ollama

1. Go to [https://ollama.com/download](https://ollama.com/download)
2. Download and install the version for your operating system (macOS, Windows, or Linux)
3. After installation, open a terminal (command line) and verify it works:

```
ollama --version
```

### 1.2 Downloading a model

Ollama provides a library of pre-packaged models. To download a model, use the `pull`
command in the terminal:

```
ollama pull llama3.2
```

This downloads Meta's Llama 3.2 model with 3 billion parameters (roughly 2 GB).
You can browse all available models at [https://ollama.com/library](https://ollama.com/library).

Some recommended models for this course:

| Model | Size | Command |
|---|---|---|
| Llama 3.2 (3B) | ~2 GB | `ollama pull llama3.2` |
| Gemma 3 (4B) | ~3 GB | `ollama pull gemma3:4b` |
| Gemma 3 (1B) | ~815 MB | `ollama pull gemma3:1b` |

You can see all downloaded models with:

```
ollama list
```

### 1.3 Testing a model in the terminal

Before connecting from Python, you can test a model directly in the terminal:

```
ollama run llama3.2 "What is sentiment analysis? Answer in one sentence."
```

This confirms the model is downloaded and working.

### 1.4 Connecting to Ollama from Python

When Ollama is running, it exposes an API on `http://localhost:11434` that is compatible
with the OpenAI API format. This means we can use the `openai` Python package to connect
to it. The advantage of this approach is that the same code works with cloud APIs (OpenAI,
etc.) by simply changing the `base_url` and `api_key`.

Install the package if needed:

In [1]:
# !pip install openai

Now we connect to Ollama and define the model we want to use. If you want to switch
to a different model, simply change the `MODEL` variable below.

In [2]:
from openai import OpenAI

# --- Configuration ---
# Change the model name here to use a different model
MODEL = "llama3.2"

# Connect to the local Ollama server
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1/",
    api_key="ollama"  # required by the library but ignored by Ollama
)

We define a helper function that sends a prompt and returns the response text. This
keeps the rest of the notebook clean.

In [5]:
def ask_llm(prompt):
    """Send a prompt to the local LLM and return the response text."""
    response = ollama_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [6]:
# Quick test
print(ask_llm("What is sentiment analysis? Answer in one sentence."))

Sentiment analysis, also known as opinion mining or emotional intelligence, is the process of automatically determining the emotional tone or attitude conveyed by a piece of text, such as a review, comment, or social media post.


## 2. Sentiment Analysis

Sentiment analysis aims to determine the emotional tone expressed in a piece of text, for
example whether a product review is positive, negative, or neutral.

Traditionally this required building NLP pipelines with tokenisation, feature extraction and
trained classifiers. With an LLM, we can instead describe the task in a prompt and let the
model handle the classification.

### 2.1 Sample data

We create a small DataFrame of customer reviews for a fictional electronics retailer.

In [7]:
import pandas as pd

reviews = pd.DataFrame({
    "review_id": [1, 2, 3, 4, 5],
    "product": [
        "Wireless Headphones",
        "USB-C Hub",
        "Mechanical Keyboard",
        "Portable Charger",
        "Webcam HD Pro"
    ],
    "review_text": [
        "Absolutely love these headphones! The noise cancellation is incredible and the battery lasts all day.",
        "Stopped working after two weeks. Total waste of money. Would not recommend to anyone.",
        "Decent keyboard for the price. Keys feel a bit mushy but overall it gets the job done.",
        "Fast charging and compact design. Exactly what I needed for travel. Five stars!",
        "The image quality is grainy and the microphone picks up a lot of background noise. Very disappointed."
    ]
})

reviews

,review_id,product,review_text
0,1,Wireless Headphones,Absolutely love these headphones! The noise ca...
1,2,USB-C Hub,Stopped working after two weeks. Total waste o...
2,3,Mechanical Keyboard,Decent keyboard for the price. Keys feel a bit...
3,4,Portable Charger,Fast charging and compact design. Exactly what...
4,5,Webcam HD Pro,The image quality is grainy and the microphone...


### 2.2 Analysing sentiment

We construct a prompt that asks the model to classify each review as Positive, Negative or
Neutral. The key to getting structured, usable results is to be explicit about the expected
output format in the prompt.

In [8]:
def analyse_sentiment(text):
    """Classify the sentiment of a single review as Positive, Negative or Neutral."""
    prompt = f"""Classify the sentiment of the following customer review.
Respond with exactly one word: Positive, Negative, or Neutral.

Review: {text}"""
    result = ask_llm(prompt)
    return result.strip()

# Test on the first review
print(analyse_sentiment(reviews.loc[0, "review_text"]))

Positive


Now we apply the function to every review in the DataFrame. Note that each call
sends a separate request to the model.

In [9]:
# Apply sentiment analysis to all reviews
reviews["sentiment"] = reviews["review_text"].apply(analyse_sentiment)

reviews[["product", "review_text", "sentiment"]]

,product,review_text,sentiment
0,Wireless Headphones,Absolutely love these headphones! The noise ca...,Positive
1,USB-C Hub,Stopped working after two weeks. Total waste o...,Negative
2,Mechanical Keyboard,Decent keyboard for the price. Keys feel a bit...,Neutral
3,Portable Charger,Fast charging and compact design. Exactly what...,Positive
4,Webcam HD Pro,The image quality is grainy and the microphone...,Negative


### 2.3 Extracting more detail

We can also ask the model to return richer output with a short
explanation. When doing this, it helps to ask for a structured format like JSON.

In [10]:
import json

def analyse_sentiment_detailed(text):
    """Return sentiment label and a brief explanation."""
    prompt = f"""Analyse the sentiment of the following customer review.
Return your answer as a JSON object with two keys:
- "sentiment": one of "Positive", "Negative", or "Neutral"
- "explanation": a one-sentence explanation

Do not include any other text, markdown formatting, or code blocks. Return only the JSON object.

Review: {text}"""
    result = ask_llm(prompt)
    # Clean the response in case the model wraps it in markdown code fences
    cleaned = result.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(cleaned)

# Test on one review
detail = analyse_sentiment_detailed(reviews.loc[1, "review_text"])
print(json.dumps(detail, indent=2))

{
  "sentiment": "Negative",
  "explanation": "The reviewer was extremely dissatisfied with their purchase, stating it stopped working shortly after use and would not recommend it to anyone."
}


## 3. Text Classification

Text classification assigns predefined categories or labels to a piece of text. Sentiment
analysis is technically a specific case of text classification, but the term usually refers
to broader categorisation tasks such as topic labelling or priority assignment.

With an LLM, we define the categories in the prompt and the model classifies each text
accordingly.

### 3.1 Sample data

We create a small dataset of customer support messages for a fictional software company.

In [11]:
tickets = pd.DataFrame({
    "ticket_id": [101, 102, 103, 104, 105],
    "message": [
        "I cannot log in to my account. I have tried resetting my password three times but the reset email never arrives.",
        "Would it be possible to add dark mode to the mobile app? Many of us use the app at night.",
        "I was charged twice for my subscription this month. Please issue a refund for the duplicate payment.",
        "The app crashes every time I try to export a report to PDF. This has been happening since the last update.",
        "Just wanted to say that the new dashboard design is fantastic. Great work by the team!"
    ]
})

tickets

,ticket_id,message
0,101,I cannot log in to my account. I have tried re...
1,102,Would it be possible to add dark mode to the m...
2,103,I was charged twice for my subscription this m...
3,104,The app crashes every time I try to export a r...
4,105,Just wanted to say that the new dashboard desi...


### 3.2 Classifying tickets by category

We define a set of categories and ask the model to assign the most appropriate one to each
message.

In [20]:
CATEGORIES = ["Account Issue", "Feature Request", "Billing", "Bug Report", "Feedback"]

def classify_ticket(text):
    """Classify a support ticket into one of the predefined categories."""
    categories_str = ", ".join(CATEGORIES)
    prompt = f"""Classify the following customer support message into exactly one of these categories: {categories_str}.
Respond with only the category name, nothing else.

Message: {text}"""
    result = ask_llm(prompt)
    return result.strip()

# Test on the first ticket
print(tickets.loc[0, "message"])
print()
print(classify_ticket(tickets.loc[0, "message"]))

I cannot log in to my account. I have tried resetting my password three times but the reset email never arrives.

Account Issue


In [21]:
# Apply to all tickets
tickets["category"] = tickets["message"].apply(classify_ticket)

tickets[["ticket_id", "category"]]

,ticket_id,category
0,101,Account Issue
1,102,Feature Request
2,103,Account Issue
3,104,Bug Report
4,105,Feedback


### 3.3 Adding priority levels

We can extend the classification to also assign a priority level. Again, being explicit
about the expected format helps produce clean, parseable results.

In [27]:
def classify_ticket_detailed(text):
    """Classify a ticket by category and priority, returning a JSON object."""
    categories_str = ", ".join(CATEGORIES)
    prompt = f"""Classify the following customer support message.
Return a JSON object with two keys:
- "category": one of {categories_str}
- "priority": one of "High", "Medium", "Low"

Do not include any other text, markdown formatting, or code blocks. Return only the JSON object.

Message: {text}"""
    result = ask_llm(prompt)
    cleaned = result.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(cleaned)

# Test on the billing ticket
detail = classify_ticket_detailed(tickets.loc[2, "message"])
print(json.dumps(detail, indent=2))

{
  "category": "Account Issue",
  "priority": "High"
}


In [28]:
# Apply detailed classification to all tickets
details = tickets["message"].apply(classify_ticket_detailed)

tickets["category"] = details.apply(lambda d: d.get("category", ""))
tickets["priority"] = details.apply(lambda d: d.get("priority", ""))

tickets[["ticket_id", "message", "category", "priority"]]

,ticket_id,message,category,priority
0,101,I cannot log in to my account. I have tried re...,Account Issue,High
1,102,Would it be possible to add dark mode to the m...,Feature Request,High
2,103,I was charged twice for my subscription this m...,Billing,High
3,104,The app crashes every time I try to export a r...,Bug Report,High
4,105,Just wanted to say that the new dashboard desi...,Feature Request,High


## 4. Important considerations

A few things to keep in mind when using LLMs for text analysis:

**Non-determinism.** LLMs are probabilistic. Running the same prompt twice may produce
slightly different results. For tasks where consistency matters, setting `temperature=0`
in the API configuration can help (though it does not guarantee identical outputs).

**Validation.** Always validate the model's output before using it downstream. The JSON
parsing in the examples above can fail if the model returns unexpected formatting. In
production code, wrap parsing in try/except blocks.

**Model quality.** Smaller local models may struggle with complex classification tasks or
produce inconsistent JSON formatting. If you notice issues, try a larger model (e.g.
`gemma3:12b` instead of `gemma3:4b`) or switch to a cloud API.

## 5. Using a cloud API instead (Gemini)

If your hardware cannot run local models comfortably, or if you need a more capable model,
you can use a cloud LLM API instead. The code structure is almost identical.

Below is an example using Google's Gemini API. You will need a Gemini API key stored in
your `.env` file (refer to the Week 3 exercise instructions).

In [31]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.environ.get("GEMINI_API_KEY")

if api_key:
    print(f"Key loaded: {api_key[:5]}...")
else:
    print("ERROR: GEMINI_API_KEY not found. Check your .env file.")

Key loaded: AIzaS...


In [32]:
from google import genai

gemini_client = genai.Client(api_key=api_key)

# Change the model here if needed
GEMINI_MODEL = "gemini-2.5-flash-lite"

def ask_gemini(prompt):
    """Send a prompt to Gemini and return the response text."""
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt
    )
    return response.text

In [33]:
# Quick test
print(ask_gemini("What is sentiment analysis? Answer in one sentence."))

Sentiment analysis is the process of computationally identifying and categorizing opinions expressed in a piece of text, especially to determine whether the writer's attitude towards a particular topic, product, etc., is positive, negative, or neutral.


To use Gemini for the sentiment analysis and classification examples from Sections
2 and 3, simply replace `ask_llm` with `ask_gemini` in the function definitions. The
prompts and logic remain exactly the same.

Note that Gemini's free tier has a limited number of tokens per day. You run out of free
tokens quite fast when experimenting. If your hardware is good enough, using Ollama locally
is the more practical option for this course.

### Local vs. cloud models

| Aspect | Cloud API (e.g. Gemini) | Local model (e.g. Ollama) |
|---|---|---|
| **Quality** | State-of-the-art models | Smaller, less capable models |
| **Cost** | Pay per token (limited free tier) | Free after download |
| **Privacy** | Data sent to external servers | Data stays on your machine |
| **Hardware** | No special requirements | Needs sufficient RAM (8 GB+) and ideally a GPU |
| **Setup** | API key only | Install software and download models |
| **Speed** | Fast (server-side GPUs) | Depends on your hardware |